# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [ ]:
VARIANT = None   # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [ ]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [ ]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

#### ✍️ Дизайн-документ

Заполните правую колонку для своей акции — как в разделе 8 семинара С7 и на слайде 32 лекции Л7. Многоточия замените своим текстом.


| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | Если подписчики будут получать еженедельную подборку партнёрских скидок через push-уведомления, средний оборот в партнерских сервисах вырастет не менее чем на 3% (примерно 95 ₽ на клиента) за 4 недели. Ожидаемый механизм - повышение интереса к партнёрским предложениям и увеличение количества покупок. |
| единица рандомизации и почему именно она (подпиской делятся — колонка `members`) | Подписка целиком, представленная владельцем (плательщиком). Все участники одной подписки должны находиться в одной группе, чтобы исключить передачу предложений и взаимное влияние теста и контроля. |
| группы и доли | Control - 50% клиентов, обычные условия без новой рассылки. Test - 50% клиентов, еженедельная push-подборка партнёрских скидок. Случайное распределение 1:1. |
| основная метрика | Средний оборот в партнерских сервисах на одного клиента за 4 недели (`spend`), в рублях. Клиенты без покупок учитываются с нулевым оборотом. |
| метрики-стражи: что не должно ухудшиться | Отток подписчиков (`churned`) и количество обращений в поддержку (`tickets`). Акция не должна увеличивать отток и создавать дополнительную нагрузку на поддержку. |
| α, мощность, MDE - числами из ячейки выше | α = 0.05; мощность = 0.80; 3000 клиентов на группу. MDE без CUPED = 233 ₽, MDE с CUPED = 185 ₽. Ожидаемый прирост 3% = 95 ₽. |
| поправка на множественные сравнения | Поправка Холма (Holm) для одновременной проверки четырех метрик: оборота, покупок, оттока и обращений. |
| оценщик | Разность средних test - control с 95% доверительным интервалом и t-критерием Уэлча. Основная оценка точности - CUPED с использованием оборота за предыдущие 4 недели (`spend_prev`). Анализ по первоначальному назначению в группы (ITT). |
| окно | 4 недели (28 дней), единый заранее фиксированный период наблюдения для обеих групп. |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | Запускаем акцию, если эффект статистически значим, прирост составляет не менее 95 ₽, метрики-стражи не ухудшились и экономический результат положительный. Если оборот растёт, но ухудшаются защитные метрики - дорабатываем акцию и повторяем пилот. Если верхняя граница доверительного интервала ниже 95 ₽ - отказываемся от запуска в текущем виде. Если интервал слишком широкий и не позволяет принять решение - продлеваем или повторяем эксперимент. |


## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes)

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
effect = diff_ci(test["spend"], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [ ]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [ ]:
# Числа для ответа на вопрос 1 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
print(f"Размеры групп: {sizes.to_dict()}")
print(f"SRM p-value: {srm.pvalue:.4f}")

print(f"Баланс до эксперимента: test = {test['spend_prev'].mean():.2f} ₽")
print(f"Баланс до эксперимента: control = {control['spend_prev'].mean():.2f} ₽")
print(f"Баланс p-value: {balance.pvalue:.4f}")

print(f"Эффект: {effect['diff']:.2f} ₽")
print(f"95% доверительный интервал: [{effect['lo']:.2f}; {effect['hi']:.2f}]")
print(f"MDE без CUPED: {mde_raw:.2f} ₽")
print(f"MDE с CUPED: {mde_cuped:.2f} ₽")

print(f"HypEx t-test p-value: {ab['TTest p-value'].iloc[0]:.4f}")
print(f"HypEx U-test p-value: {ab['UTest p-value'].iloc[0]:.4f}")


#### ✍️ Ответ на вопрос 1

Проверка SRM не выявила перекоса в распределении клиентов между группами (p = 0,2249). До эксперимента тестовая и контрольная группы также были сопоставимы по обороту (p = 0,5987). После акции средний оборот в тестовой группе оказался выше на 264,85 руб, 95% доверительный интервал составил [90,31; 439.39] руб и не включает ноль. Эффект статистически значим: t-test дал p = 0,0030, U-test - p = 0,0361. Разница между p-значениями объясняется тем, что t-test сравнивает средние, а U-test - распределения. Оба критерия подтверждают статистическую значимость результата.


## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [ ]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table).pvalue

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [ ]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [ ]:
# Ваша cuped() из pilot.py: метрика spend, ковариата spend_prev -> mine["spend_cuped"] и theta.
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [ ]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole(parent="spend", lag=1)

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [ ]:
# Числа для ответа на вопрос 2 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.

# 1. Защитные метрики
print("ЗАЩИТНЫЕ МЕТРИКИ")
for metric in ("churned", "tickets"):
    row = guards.loc[metric]
    print(f"{metric}: diff = {row['diff']:.4f}, p = {row['p']:.4f}")

print(f"Chi-square по оттоку: p = {p_churn:.4f}")

# 2. Что осталось значимым после поправки Холма
print("\nПОПРАВКА ХОЛМА")
holm_significant = metrics.loc[metrics["holm"] < 0.05, "feature"].tolist()
print("Значимые метрики:", holm_significant)

# 3. Сравнение эффекта до и после CUPED
raw_diff = float(effect["diff"])
cuped_diff = float(effect_cuped["diff"])

print("\nЭФФЕКТ")
print(f"До CUPED: {raw_diff:.2f} ₽")
print(f"После CUPED: {cuped_diff:.2f} ₽")
print(f"Изменение оценки: {cuped_diff - raw_diff:.2f} ₽")
print(f"p после CUPED: {effect_cuped['p']:.4f}")

# 4. Сравнение доверительных интервалов
width_raw = float(effect["hi"] - effect["lo"])
width_cuped = float(effect_cuped["hi"] - effect_cuped["lo"])
narrowing = (width_raw - width_cuped) / width_raw * 100

print("\nДОВЕРИТЕЛЬНЫЕ ИНТЕРВАЛЫ")
print(f"До CUPED: {width_raw:.2f} ₽")
print(f"После CUPED: {width_cuped:.2f} ₽")
print(f"Сужение интервала: {narrowing:.2f}%")

# 5. Сравнение оборота до эксперимента
prev_test = float(test["spend_prev"].mean())
prev_control = float(control["spend_prev"].mean())

print("\nПРЕДЭКСПЕРИМЕНТАЛЬНЫЙ БАЛАНС")
print(f"Test: {prev_test:.2f} ₽")
print(f"Control: {prev_control:.2f} ₽")
print(f"Разница: {prev_test - prev_control:.2f} ₽")
print(f"p-value: {balance.pvalue:.4f}")


#### ✍️ Ответ на вопрос 2


Защитные метрики статистически значимо не ухудшились: отток вырос на 0,87 пп (p = 0,0602), а обращения в поддержку снизились на 0,21 пп (p = 0,7344). После поправки Холма значимыми остались оборот (spend) и покупки (purchases). CUPED скорректировал оценку эффекта с +264,85 до +236,65 руб, но эффект остался положительным и статистически значимым: 95% ДИ [97,30; 376,00] руб. Ширина доверительного интервала сократилась с 349,08 до 278,70 ₽, то есть на 20,16%. До эксперимента оборот тестовой группы составлял 3181,17 руб против 3137,31 руб в контроле; разница 43,86 ₽ статистически незначима (p = 0,5987). CUPED учёл исходный уровень трат клиентов и уменьшил влияние межклиентских различий на оценку эффекта.


## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [ ]:
# Задача 9. 95 % интервал разности средних оборота test − control бутстрепом:
# 2 000 повторов, rng = np.random.default_rng(VARIANT), сначала тест, потом контроль.



rng = np.random.default_rng(VARIANT)

test_spend = test["spend"].to_numpy()
control_spend = control["spend"].to_numpy()

boot_diffs = []

for _ in range(2000):
    # Сначала пересобираем тестовую группу
    boot_test = rng.choice(
        test_spend,
        size=len(test_spend),
        replace=True
    )

    # Затем контрольную группу
    boot_control = rng.choice(
        control_spend,
        size=len(control_spend),
        replace=True
    )

    # Разница средних оборотов
    boot_diffs.append(boot_test.mean() - boot_control.mean())

# Нижняя и верхняя границы 95% интервала
answer_9 = [
    round(float(x), 2)
    for x in np.percentile(boot_diffs, [2.5, 97.5])
]

answer_9


### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [ ]:
# Задача 10. ρ² с spend (в %) для шести кандидатов из предпериода и отдельно для purchases.
# Ответ — лучший кандидат из предпериода и его ρ² в процентах.


# Задача 10. Сравнение шести ковариат для CUPED

candidates = {
    "spend_prev": mine["spend_prev"],
    "spend_prev2": mine["spend_prev2"],
    "spend_prev+spend_prev2": mine["spend_prev"] + mine["spend_prev2"],
    "purchases_prev": mine["purchases_prev"],
    "sessions_prev": mine["sessions_prev"],
    "sessions_prev2": mine["sessions_prev2"],
}

# Считаем квадрат корреляции с текущим оборотом в процентах
rho2 = {}

for name, x in candidates.items():
    pair = pd.concat(
        [mine["spend"].rename("spend"), x.rename("candidate")],
        axis=1
    ).dropna()

    correlation = pair["spend"].corr(pair["candidate"])
    rho2[name] = float(correlation ** 2 * 100)

# Сортируем кандидатов от лучшего к худшему
rho2_table = pd.Series(rho2, name="rho², %").sort_values(ascending=False)

print("ШЕСТЬ КАНДИДАТОВ:")
print(rho2_table.round(1).to_string())

# Отдельно считаем purchases — это ловушка, НЕ кандидат
purchase_pair = mine[["spend", "purchases"]].dropna()
purchase_corr = purchase_pair["spend"].corr(purchase_pair["purchases"])
purchase_rho2 = purchase_corr ** 2 * 100

print(f"\nPurchases (во время эксперимента): {purchase_rho2:.1f}%")

# Ответ — лучший из шести признаков предпериода
best_covariate = rho2_table.index[0]

answer_10 = [
    str(best_covariate),
    round(float(rho2_table.loc[best_covariate]), 1)
]

print("\nОТВЕТ:")
answer_10


### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
# Задача 11. Эффект на spend мэтчингом: Matching(n_neighbors=5),
# признаки age, tenure_days, kyc_level, members, spend_prev, sessions_prev.
# Ответ — ATE, нижняя и верхняя граница интервала.


# Задача 11. Эффект на spend методом Matching

# 1. Кодируем тест и контроль числами
mine["t"] = (mine["grp"] == "test").astype(int)

# 2. Признаки, измеренные ДО эксперимента
features_11 = [
    "age",
    "tenure_days",
    "kyc_level",
    "members",
    "spend_prev",
    "sessions_prev",
]

# 3. Назначаем роли HypEx
roles_11 = {
    "client_id": InfoRole(str),
    "t": TreatmentRole(),
    "spend": TargetRole(),
    **{feature: FeatureRole() for feature in features_11},
}

# 4. Создаём Dataset и запускаем мэтчинг
ds_11 = Dataset(
    roles=roles_11,
    data=mine,
    default_role=InfoRole(),
)

res_11 = Matching(n_neighbors=5).execute(ds_11)

# 5. Выводим результаты HypEx
matching_summary = F(res_11.resume)
print(matching_summary.to_string())

# 6. Забираем ATE и его 95% доверительный интервал
matching_summary.index = matching_summary.index.astype(str)
ate = matching_summary.loc["ATE"]

answer_11 = [
    round(float(ate["Effect Size"]), 2),
    round(float(ate["CI Lower"]), 2),
    round(float(ate["CI Upper"]), 2),
]

print("\nОТВЕТ:")
answer_11


### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [ ]:
# Задача 12. Результат акции в рублях на клиента за 4 недели.
# Допущения — не меняйте их.
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнёрских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, ₽
CHURN_COST = 750      # потеря от ушедшего подписчика: три месяца «Базового», ₽


# Доход от комиссии с дополнительного оборота
commission_income = float(effect_cuped["diff"]) * COMMISSION

# Экономия на обращениях в поддержку
ticket_diff = float(guards.loc["tickets", "diff"])
ticket_saving = -ticket_diff * TICKET_COST

# Потери из-за дополнительного оттока
churn_diff = float(guards.loc["churned", "diff"])
churn_loss = -churn_diff * CHURN_COST

# Итог на одного клиента за четыре недели
answer_12 = round(
    commission_income + ticket_saving + churn_loss, 2
)

print(f"Комиссия с прироста оборота: +{commission_income:.2f} ₽")
print(f"Экономия на обращениях: {ticket_saving:+.2f} ₽")
print(f"Эффект дополнительного оттока: {churn_loss:+.2f} ₽")
print(f"ИТОГО на клиента: {answer_12:.2f} ₽")

answer_12


**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [ ]:
# Числа для ответа на вопрос 3 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.

# Вопрос 3. Сравнение методов и экономики акции

# 1. Доверительные интервалы
intervals = {
    "Обычный метод (задача 3)": (
        float(effect["lo"]), float(effect["hi"])
    ),
    "CUPED (задача 7)": (
        float(effect_cuped["lo"]), float(effect_cuped["hi"])
    ),
    "Bootstrap (задача 9)": (
        float(answer_9[0]), float(answer_9[1])
    ),
    "Matching (задача 11)": (
        float(answer_11[1]), float(answer_11[2])
    ),
}

print("ДОВЕРИТЕЛЬНЫЕ ИНТЕРВАЛЫ И ИХ ШИРИНА")

widths = {}

for name, (lo, hi) in intervals.items():
    widths[name] = hi - lo
    print(
        f"{name}: [{lo:.2f}; {hi:.2f}], "
        f"ширина = {widths[name]:.2f} ₽"
    )

# 2. Сравнение оценок эффекта
print("\nОЦЕНКИ ЭФФЕКТА")
print(f"Обычная оценка: {effect['diff']:.2f} ₽")
print(f"CUPED: {effect_cuped['diff']:.2f} ₽")
print(f"Bootstrap: средний эффект {effect['diff']:.2f} ₽")
print(f"Matching ATE: {answer_11[0]:.2f} ₽")

# 3. Ковариаты
print("\nКОВАРИАТЫ")
print(f"Лучшая допустимая: {answer_10[0]}")
print(f"Её rho²: {answer_10[1]:.1f}%")
print(f"Purchases во время теста: {purchase_rho2:.1f}%")

# 4. Экономика
print("\nЭКОНОМИКА")
print(f"Комиссия: {commission_income:+.2f} ₽")
print(f"Экономия на поддержке: {ticket_saving:+.2f} ₽")
print(f"Эффект от оттока: {churn_loss:+.2f} ₽")
print(f"ИТОГ: {answer_12:.2f} ₽ на клиента")

print("\nШИРИНЫ ИНТЕРВАЛОВ:")
print({k: round(float(v), 2) for k, v in widths.items()})


#### ✍️ Ответ на вопрос 3


Больше всего доверяю оценке CUPED: она использует оборот до эксперимента и даёт самый узкий 95% доверительный интервал [97,30; 376,00] руб шириной 278.70 руб. Для сравнения, ширина обычного интервала составляет 349.09 руб, бутстрепа - 362.57 руб, мэтчинга - 285.95 ₽. Purchases нельзя использовать как ковариату, несмотря на высокое ρ² = 69.5%, поскольку покупки измерены во время эксперимента и сами могли измениться под влиянием акции, что исказило бы оценку эффекта. При допущениях задачи 12 акция не окупается: результат составляет -2.66 руб на клиента за четыре недели; при этом в расчёте не учтены реальные расходы на подготовку и отправку push-уведомлений, а также возможные долгосрочные эффекты акции


## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

#### ✍️ Отчёт продакту


**Что проверяли:** Оценивали эффективность еженедельной push-подборки партнёрских скидок для клиентов ПРАЙМ. Проверяли, увеличивает ли акция средний оборот в партнёрских сервисах за четыре недели. В эксперименте участвовали 6000 клиентов, распределенных между тестовой и контрольной группами.

**Что получилось:** CUPED-оценка прироста оборота составила 236,65 рублев на клиента за четыре недели, 95% доверительный интервал [97,30; 376,00] руб. Эффект положительный и статистически значимый. После поправки Холма значимыми остались оборот и покупки. Статистически значимого ухудшения метрик-стражей не обнаружено, однако отток в тестовой группе выше на 0,87 пп Финансовый результат с учётом заданных допущений отрицательный: −2,66 рублев на клиента.

**Какие остались вопросы:** Эксперимент длился только четыре недели, поэтому мы не знаем долгосрочного влияния акции на поведение и удержание клиентов. Отток показал пограничный сигнал ухудшения, который требует дополнительного наблюдения. Финансовая оценка основана на условных тарифах и не учитывает все возможные расходы на подготовку и отправку push-уведомлений

**Что рекомендуем:** Не запускать акцию на всю аудиторию в текущем виде. Несмотря на статистически значимый рост оборота, при заданных экономических допущениях акция пока не окупается. Рекомендуем доработать механику партнёрских скидок и снизить стоимость коммуникации, отдельно уделив внимание удержанию клиентов. После этого провести повторный эксперимент и проверить финансовый результат.


## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [ ]:
# Исследование А: ваш код. Всё — от mine.


from hypex import Matching
from hypex.dataset import FeatureRole

# 1. Определяем активных клиентов по медиане предпериода
sessions_median = mine["sessions_prev"].median()

active_mask = mine["sessions_prev"] >= sessions_median

# 2. Создаём наблюдательные данные:
# из Test оставляем активных, Control оставляем полностью
observational = mine.loc[
    (mine["grp"] == "control") |
    ((mine["grp"] == "test") & active_mask)
].copy()

observational["t"] = (
    observational["grp"] == "test"
).astype(int)

print("1. НАБЛЮДАТЕЛЬНЫЕ ДАННЫЕ")
print(f"Медиана sessions_prev: {sessions_median:.2f}")
print(f"Всего наблюдений: {len(observational)}")
print(f"Test: {(observational['t'] == 1).sum()}")
print(f"Control: {(observational['t'] == 0).sum()}")

# 3. Наивная разность средних
naive_effect = (
    observational.loc[observational["t"] == 1, "spend"].mean()
    -
    observational.loc[observational["t"] == 0, "spend"].mean()
)

print(f"\n2. НАИВНЫЙ ЭФФЕКТ: {naive_effect:.2f} ₽")

# 4. Эталон: активные клиенты обеих исходных групп
active_test = mine.loc[
    (mine["grp"] == "test") & active_mask, "spend"
]
active_control = mine.loc[
    (mine["grp"] == "control") & active_mask, "spend"
]

benchmark = active_test.mean() - active_control.mean()

print("\n3. ЭТАЛОН ПО РАНДОМИЗИРОВАННЫМ ДАННЫМ")
print(f"Активных в Test: {len(active_test)}")
print(f"Активных в Control: {len(active_control)}")
print(f"Эталонный эффект: {benchmark:.2f} ₽")

# 5. Признаки для мэтчинга — все шесть из задачи 11
features_a = [
    "age",
    "tenure_days",
    "kyc_level",
    "members",
    "spend_prev",
    "sessions_prev",
]

def run_matching_a(data, features, label):
    roles = {
        "client_id": InfoRole(str),
        "t": TreatmentRole(),
        "spend": TargetRole(),
        **{feature: FeatureRole() for feature in features}
    }

    result = Matching(n_neighbors=5).execute(
        Dataset(
            roles=roles,
            data=data,
            default_role=InfoRole()
        )
    )

    summary = F(result.resume)
    summary.index = summary.index.astype(str)

    print(f"\n{label}")
    print(summary.to_string())

    return summary.loc["ATT"]

# 6. Мэтчинг со всеми шестью признаками
att_full = run_matching_a(
    observational,
    features_a,
    "4. ATT МЭТЧИНГОМ — ВСЕ ШЕСТЬ ПРИЗНАКОВ"
)

# 7. Повторяем без sessions_prev
features_without_sessions = [
    feature for feature in features_a
    if feature != "sessions_prev"
]

att_without_sessions = run_matching_a(
    observational,
    features_without_sessions,
    "5. ATT МЭТЧИНГОМ — БЕЗ sessions_prev"
)

# 8. Сводим результаты
research_a_results = pd.DataFrame(
    {
        "estimate": [
            naive_effect,
            benchmark,
            float(att_full["Effect Size"]),
            float(att_without_sessions["Effect Size"])
        ]
    },
    index=[
        "Naive",
        "Randomized benchmark",
        "Matching with sessions_prev",
        "Matching without sessions_prev"
    ]
)

print("\n6. ИТОГОВОЕ СРАВНЕНИЕ")
print(research_a_results.round(2).to_string())


#### ✍️ Исследование А


**Вопрос:** насколько сильно смещается оценка эффекта, если акцию показывать только активным клиентам, а не проводить рандомизацию?

**Как проверяли:** сформировали наблюдательную выборку: в тест включили только клиентов с sessions_prev не ниже медианы 15, а контроль оставили целиком. Сравнили наивную разность средних, эталонный эффект среди активных клиентов обеих рандомизированных групп и результаты мэтчинга с шестью признаками и без sessions_prev

**Что получилось:** наивная оценка составила 1809,68 руб против эталонных 476.18 руб. Мэтчинг со всеми шестью признаками дал ATT = 384.42 руб, то есть приблизился к эталону. После исключения sessions_prev оценка выросла до 890,47 руб и снова значительно отклонилась от эталон

**Что это значит:** мэтчинг способен уменьшить смещение, если учитывать признаки, по которым происходил отбор клиентов. Если исключить важный признак активности, качество оценки ухудшается. Однако этот расчёт не доказывает отсутствие смещения по ненаблюдаемым признакам и не заменяет рандомизированный эксперимент.


### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [ ]:
# Исследование Б: ваш код. Всё — от mine.



from statistics import NormalDist
from math import ceil

# 1. Берём только контрольную группу — здесь эффекта акции нет
control_pair = control[["spend", "spend_prev"]].dropna()

y = control_pair["spend"]
x = control_pair["spend_prev"]

# 2. Считаем CUPED на контрольной группе
y_cuped, theta_b = cuped(y, x)

raw_values = y.to_numpy()
cuped_values = y_cuped.to_numpy()

# Целевой эффект — 3% от среднего оборота контроля
delta = float(y.mean()) * 0.03

print(f"Средний оборот контроля: {y.mean():.2f} ₽")
print(f"Эффект 3%: {delta:.2f} ₽")
print(f"Theta CUPED: {theta_b:.3f}")
print(f"Снижение дисперсии: {1 - y_cuped.var() / y.var():.1%}")

# 3. Симуляция мощности
rng = np.random.default_rng(VARIANT)

n_values = [1000, 3000, 10000, 20000, 40000]
repetitions = 500
alpha = 0.05

def simulate_power(raw, adjusted, n, rng):
    raw_hits = 0
    cuped_hits = 0

    for _ in range(repetitions):
        # Выбираем 2n клиентов с возвращением
        indices = rng.choice(
            len(raw),
            size=2 * n,
            replace=True
        )

        # Один и тот же набор клиентов для обоих методов
        raw_sample = raw[indices]
        cuped_sample = adjusted[indices]

        # Первая половина — тест с эффектом +3%
        # Вторая половина — контроль без эффекта
        p_raw = stats.ttest_ind(
            raw_sample[:n] + delta,
            raw_sample[n:],
            equal_var=False
        ).pvalue

        p_cuped = stats.ttest_ind(
            cuped_sample[:n] + delta,
            cuped_sample[n:],
            equal_var=False
        ).pvalue

        raw_hits += p_raw < alpha
        cuped_hits += p_cuped < alpha

    return raw_hits / repetitions, cuped_hits / repetitions


# 4. Считаем мощность для пяти размеров выборки
rows = []

for n in n_values:
    power_raw, power_cuped = simulate_power(
        raw_values,
        cuped_values,
        n,
        rng
    )

    rows.append({
        "n_per_group": n,
        "power_without_CUPED": power_raw,
        "power_with_CUPED": power_cuped
    })

power_table = pd.DataFrame(rows)

print("\nМОЩНОСТЬ ПО СИМУЛЯЦИИ")
print(power_table.round(3).to_string(index=False))

# 5. Считаем объём по формуле семинара С7
z = NormalDist().inv_cdf

sigma_raw = float(np.std(raw_values, ddof=1))
sigma_cuped = float(np.std(cuped_values, ddof=1))

def required_n(sigma, effect, alpha=0.05, power=0.8):
    return ceil(
        2 * (
            z(1 - alpha / 2) + z(power)
        ) ** 2 * sigma ** 2 / effect ** 2
    )

n_formula_raw = required_n(sigma_raw, delta)
n_formula_cuped = required_n(sigma_cuped, delta)

print("\nОБЪЁМ ПО ФОРМУЛЕ С7")
print(f"Без CUPED: {n_formula_raw} клиентов на группу")
print(f"С CUPED: {n_formula_cuped} клиентов на группу")

# 6. Находим первый размер, где симуляция достигает 80%
def first_n_at_80(column):
    reached = power_table.loc[
        power_table[column] >= 0.80,
        "n_per_group"
    ]

    return int(reached.iloc[0]) if len(reached) else "Больше 40000"

n_sim_raw = first_n_at_80("power_without_CUPED")
n_sim_cuped = first_n_at_80("power_with_CUPED")

print("\nГДЕ ДОСТИГАЕМ МОЩНОСТИ 80%")
print(f"Без CUPED: {n_sim_raw}")
print(f"С CUPED: {n_sim_cuped}")

print("\nТЕКУЩИЙ ЭКСПЕРИМЕНТ")
print(f"Test: {len(test)} клиентов")
print(f"Control: {len(control)} клиентов")

print("\nГОТОВО")


#### ✍️ Исследование Б


**Вопрос:** сколько клиентов необходимо, чтобы обнаружить рост оборота на 3% с мощностью 80%, и насколько помогает CUPED?

**Как проверяли:** взяли оборот контрольной группы, где эффекта акции нет, и провели по 500 симуляций для каждого размера группы: 1000, 3000, 10000, 20000 и 40000 клиентов. В тестовую группу добавляли эффект 3% от среднего оборота контроля (99,59 руб), затем сравнивали группы t-критерием Уэлча. Повторили расчёт с CUPED по предыдущему обороту spend_prev.

**Что получилось:** без CUPED мощность составила 60,2% при 10 000 клиентах и 86,4% при 20 000. С CUPED - 74,4% и 95,6% соответственно. На первой проверенной точке с мощностью выше 80% в обоих случаях - 20 000 клиентов на группу. По формуле С7 требуется 16 920 клиентов без CUPED и 11 182 с CUPED. Поправка снизила дисперсию на 33,9%.

**Что это значит:** CUPED повышает мощность и позволяет обнаруживать эффект при меньшей выборке. Однако текущего эксперимента (2953 клиента в тесте и 3047 в контроле) недостаточно для уверенного обнаружения роста на 3%: при 3 000 клиентах мощность составляет лишь 21,6% без CUPED и 29% с CUPED. Формула и симуляция дают разные оценки из-за тяжелого хвоста распределения оборота и приближённого характера расчётов. Симуляция также ограничена 500 повторами и проверенными размерами группп
